# Git pull — sync the workspace repo before the pipeline runs

The GitHub Action (`daily_incremental_fetch.yml`, 00:00 UTC) fetches the daily NVD/EPSS/KEV delta and commits it to `main`. This notebook pulls those commits into this workspace's git folder, so `01_landing_to_bronze` sees the newest pending batch. Runs as the **first task** of the daily Lakeflow Job, before 01 → 02 → 03. Every pull is logged to `pipeline_execution_logs` (layer = `Orchestration`).

In [0]:
%run ./00_config_and_logging

In [0]:
# Pull the latest commits from GitHub into this workspace's git folder.
# Remote only ever adds data/ files (GitHub Action), so notebook edits in the
# working tree never conflict with the pull.
from datetime import datetime
from databricks.sdk import WorkspaceClient

REPO_PATH = "/Workspace/Users/naumaniqbal987@gmail.com/cyber-threat-intel"

run_id = new_run_id()
start  = datetime.now()

try:
    w = WorkspaceClient()

    # Resolve this workspace's git folder id from the workspace path (each
    # teammate's workspace has its own clone with a different id; the repos
    # LIST api does not cover folders under /Workspace/Users, but get-by-id
    # works and the repo id equals the folder's workspace object id).
    repo_id = w.workspace.get_status(REPO_PATH).object_id
    repo = w.repos.get(repo_id)
    if repo.branch != "main":
        raise RuntimeError(f"Repo is on branch '{repo.branch}' — expected 'main'.")
    print(f"Pulling {repo.path} (branch {repo.branch}, head {str(repo.head_commit_id)[:8]}) ...")

    # There is no dedicated REST "pull" verb for workspace Git folders —
    # PATCHing with the same branch updates the checkout to that branch's
    # latest remote commit, which is the documented pull equivalent.
    w.api_client.do("PATCH", f"/api/2.0/repos/{repo.id}", body={"branch": repo.branch})

    head    = w.repos.get(repo.id).head_commit_id
    batches = sorted(f.name for f in dbutils.fs.ls(f"{REPO_PATH}/data/sample_raw/incremental") if f.isDir())

    write_log(run_id, "Orchestration", "git_pull",
              f"github:{repo.provider}:main@{str(head)[:8]}", REPO_PATH,
              start, "Success")
    print(f"Pull OK — HEAD {str(head)[:8]} | newest batch on disk: {batches[-1] if batches else 'none'}")
except Exception as e:
    write_log(run_id, "Orchestration", "git_pull", "github:main", REPO_PATH,
              start, "Failure", error_message=str(e))
    raise